In [ ]:
# greet()

<!-- linkedto: _c1fb337a -->
Solveit version: **0.0.72**  
dialoghelper version: **0.1.18**  
git branch: **dev-log**  
git changes: [' M ../dutil/_modidx.py', ' M linked_msg.ipynb', ' M scheduled_run.ipynb', ' M ../nbs/00_core.ipynb']  


Hi! I'm Vic, we're going to work together during this session.

In [ ]:
#| export
import re
from fastcore.meta import delegates
from dialoghelper.stdtools import find_msg_id, read_msg, add_msg, find_var, update_msg, find_msgs

In [ ]:
import json
from functools import wraps
from random import randint
from anyio import sleep
from anyio.from_thread import start_blocking_portal
import fastcore.all as FC
from fastcore.test import *
from dialoghelper.stdtools import *

# ----

See <a href="/dialog_?name=vic%2Fdutil%2Fexplorer%2Fscheduled_run" target="_blank">vic/dutil/explorer/scheduled_run</a>

In [ ]:
def nq(cmd, *args, **kwargs):
    """Queue the run of `cmd(*args, **kwargs)` and wait until dialog state settles.
    
    Ensures that after each call, the dialog state on the Solveit server is settled
    before returning. For `run_msg`, polls asynchronously to avoid blocking kernel execution.
    
    Note: While this ensures server-side state is settled, the frontend may lag slightly
    in displaying changes. If messages don't appear immediately, refresh the page or click
    the green websocket status button. Often, a small time.sleep after this call will alleviate the issue.
    """
    res = cmd(*args, **kwargs)
    if cmd.__name__ == 'run_msg':
        ids = kwargs.get('ids') if 'ids' in kwargs else (args[0] if args else None)
        if not ids: ids = find_msg_id()
        async def poll_exe():
            while True:
                msgs = find_msgs(ids=ids, include_meta=True, include_output=False)
                if all(m.get('time_run') for m in msgs): return
                await sleep(0.05)
        with start_blocking_portal() as portal:  portal.call(poll_exe)
        return res
    if cmd.__name__ in ('add_msg', 'update_msg', 'del_msg'): return res  # Already settled by the time the call returns
    return res  # Unknown command, just return

# The problem

In [ ]:
find_msg_id(), add_msg(f"### dup {randint(0,100)}")

('_546fe428', '_088f0bf7')

### dup 80

### dup 58

In [ ]:
find_msg_id(), add_msg(f"### more dup {randint(0,100)}")

('_ef6adc15', '_cd72e21a')

('_ef6adc15', '_d8e8529c')

### more dup 37

### more dup 94

In [ ]:
nq(run_msg, '_546fe428,_ef6adc15')

'{"status":"queued"}'

When using `add_msg` in code that gets run multiple times (via run Below, above, all, or range), each execution creates a new message. This clutters the dialog with duplicate messages, forcing users to develop ad-hoc solutions to alleviate the problem.

# A partial solution

In [ ]:
#| export
_ts, _te = ('<!-- ', '#| '), (' -->', '')
def get_tag(name:str, val:str='', msg_type:str='note'):
    return f"{_ts[msg_type=='code']}{name}: {val or find_msg_id()}{_te[msg_type=='code']}"

In [ ]:
print(tag := get_tag('tag'))
test_eq(tag, f"<!-- tag: {find_msg_id()} -->")

<!-- tag: _cff3e983 -->


In [ ]:
print(tag := get_tag('linked', '_asdfqwer'))
test_eq(tag, f"<!-- linked: _asdfqwer -->")

<!-- linked: _asdfqwer -->


In [ ]:
print(tag := get_tag('linked', '_asdfqwer', 'code'))
test_eq(tag, '#| linked: _asdfqwer')

#| linked: _asdfqwer


In [ ]:
#| export
_tagpats = {
    'code': (re.compile(r'\A#\|\W*(\w+): ([_a-f0-9]{9})\W*$', re.MULTILINE),),
    'note': (re.compile(r'\A<!-- (\w+): ([_a-f0-9]{9}) -->\W*$', re.MULTILINE),)
}
_tagpats['raw'] = _tagpats['prompt'] = _tagpats['note']
_tagpats[None] = _tagpats['note'] + _tagpats['code']

def has_tag(s:str, msg_type:str=None) -> bool:
    "Check if string contains tags created by get_tag at the start of the string"
    if msg_type == 'code' and not s.startswith('#|'): return False
    if msg_type == 'note' and not s.startswith('<!--'): return False
    return any(re.search(pat, s) for pat in _tagpats[msg_type])

This will:
- Match tags that start at the very beginning of the string (`\A`)
- Allow optional whitespace around the tag name and after the message ID (`\W*`)
- Match up to the end of the first line (`$` with `re.MULTILINE`)
- Allow any content on subsequent lines

So it will match:

In [ ]:
test_is(has_tag('<!-- tag: _abc12345 -->'), True)
test_is(has_tag('<!-- tag: _abc12345 -->\nmore content here'), True)
test_is(has_tag('#| linked: _abc12345\ncode here'), True)

But not match tags that appear later in the string or embedded in other content.

In [ ]:
test_is(has_tag('asd\n```\n<!-- tag: _abc12345 -->\nmore content here'), False)

In [ ]:
#| export
def find_tag(s: str, msg_type: str = None) -> str:
    "Find tag in a string and return their details"
    for pat in _tagpats[msg_type]:
        if match := re.search(pat, s): return f"{match.group(1)}: {match.group(2)}"
    return ''

In [ ]:
test_eq(find_tag('<!-- tag: _abc12345 -->'), 'tag: _abc12345')
test_eq(find_tag('<!-- tag: _abc12345 -->\nmore content here'), 'tag: _abc12345')
test_eq(find_tag('#| linked: _abc12345\ncode here'), 'linked: _abc12345')

In [ ]:
#| export
def get_linked(id:str) -> str:
    if (msg := read_msg(0, id=id)).get('id','') == id:
        if tag := find_tag(msg.content, msg.msg_type):
            k,v = tag.split(': ')
            if k == 'linkedto': return v
    return ''

<!-- linkedto: _abc12345 -->
asdf

In [ ]:
test_eq(get_linked(msgid := read_msg().id), '_abc12345')

In [ ]:
# def update_linked_msg(content, tagname, msgid=''):
#     msgid = msgid or find_msg_id()
#     tag = get_tag(tagname, msgid)
#     linked = find_var('__linked_msgs')
#     if reportid := linked.get(msgid):
#         msg = read_msg(0, True, reportid)
#         if 'msg' not in msg and re.search(tag, msg.content):
#             linked[msgid] = update_msg(reportid, content=f"{tag}\n{content}")
#             return
#     linked[msgid] = add_msg(f"{tag}\n{content}")

In [ ]:
#| export
get_ipython().xpush(__linked_msgs={})  # WARNING: ipykernel_helper.xpush is not documented

Note: inject `__linked_msgs` into the user namespace, not the module namespace.

In [ ]:
#| export
delegates(add_msg)
def link_msg(
    content:str=None,  # Content of the linked message
    id:str=None,  # ID of the message to link to, or current message if not provided
    **kwargs  # Additional keyword arguments for `add_msg` or `update_msg`
    ) -> str:  # id of linked message
    "Add or update a message linked to `anchor` message. Note only one linked msg per anchor."
    anchor_id, linked = id or find_msg_id(), find_var('__linked_msgs')
    if linked_id := linked.get(anchor_id):
        if (msg := read_msg(0, id=linked_id)).get('id','') == linked_id:
            tag = get_tag('linkedto', anchor_id, kwargs.get('msg_type', msg.msg_type))
            if content: kwargs['content'] = f"{tag}\n{content}"
            linked[anchor_id] = update_msg(linked_id, **kwargs)
            return linked[anchor_id]
    tag = get_tag('linkedto', anchor_id, kwargs.get('msg_type', 'note'))
    linked[anchor_id] = add_msg(f"{tag}\n{content or '.'}", id=anchor_id, **kwargs)
    return linked[anchor_id]

In [ ]:
if (get_linked(lnk := read_msg(2).id)): del_msg(lnk)

In [ ]:
linkee1 = find_msg_id()
lnkd1 = link_msg(f"### note {randint(0,100)}", pinned=True)
lnkd1

'_f85703b1'

<!-- linkedto: _519daf33 -->
### note 95

Run the above message several times. You can even reorder the linked message.

In [ ]:
test_eq(__linked_msgs, {linkee1: lnkd1})
__linked_msgs

{'_519daf33': '_f85703b1'}

In [ ]:
link_msg(f"### note {randint(0,100)}", linkee1)

'_f85703b1'

In [ ]:
test_eq(link_msg(id=linkee1, pinned=False), lnkd1)

In [ ]:
if (get_linked(lnk := read_msg(1).id)): del_msg(lnk)
linkee2 = find_msg_id()
lnkd2 = link_msg(pinned=True)
lnkd2

'_bc1cff11'

<!-- linkedto: _2cb65f60 -->
### other 11

In [ ]:
test_eq(__linked_msgs, {linkee1: lnkd1, linkee2: lnkd2})

In [ ]:
link_msg(f"### other {randint(0,100)}", linkee2)


'_bc1cff11'

In [ ]:
if (get_linked(lnk := read_msg(1).id)): del_msg(lnk)
linkee3 = find_msg_id()
lnkd3 = link_msg('2+2', msg_type='code')

In [ ]:
#| linkedto: _15848731
3-3

In [ ]:
link_msg('3-3', linkee3)

'_6637105a'

In [ ]:
test_eq(__linked_msgs, {linkee1: lnkd1, linkee2: lnkd2, linkee3: lnkd3})

In [ ]:
#| export
def hydrate():
    "Traverse dialog looking for linked messages to update `__linked_msgs`"
    linked = find_var('__linked_msgs')
    linked.clear()
    msgs = find_msgs(include_meta=True, include_output=False)
    ids = {_.id for _ in msgs}
    for msg in msgs:
        if tag := find_tag(msg.get('content', ''), msg.get('msg_type')):
            k, v = tag.split(': ')
            if k == 'linkedto' and v in ids: linked[v] = msg['id']
    return linked

Complete the `hydrate` function.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I'll complete the `hydrate` function to scan the dialog and populate `__linked_msgs` with existing linked message relationships.

<details class='tool-usage-details'>
<summary>msg_replace_lines(id='_b34345fd', start_line=1, end_line=3, new_content='def hydrate():\n    "Traverse dialog looking for li…')</summary>

```json
{
  "id": "toolu_01JN6LozJSozmRBzQGJHVAJy",
  "call": {
    "function": "msg_replace_lines",
    "arguments": {
      "id": "_b34345fd",
      "start_line": "1",
      "end_line": "3",
      "new_content": "def hydrate():\n    \"Traverse dialog looking for linked messages to update `__linked_msgs`\"\n    linked = find_var('__linked_msgs')\n    linked.clear()\n    \n    # Get all messages in the dialog\n    msgs = find_msgs(include_meta=True, include_output=False)\n    \n    # Find all messages with linkedto tags\n    for msg in msgs:\n        if tag := find_tag(msg.get('content', ''), msg.get('msg_type')):\n            k, v = tag.split(': ')\n            if k == 'linkedto':\n                # v is the anchor_id, msg['id'] is the linked message id\n                linked[v] = msg['id']\n    \n    return linked"
    }
  },
  "result": "{'success': 'Replaced lines 1 to 3 in message _b34345fd content'}"
}
```

</details>

The `hydrate` function now:

1. Gets a reference to the `__linked_msgs` global variable and clears it
2. Retrieves all messages in the dialog using `find_msgs`
3. Iterates through each message looking for tags created by `get_tag`
4. When it finds a `linkedto` tag, it extracts the anchor message ID and adds the mapping to `__linked_msgs`
5. Returns the populated dictionary

This allows you to rebuild the `__linked_msgs` state by scanning the dialog for existing linked message relationships. You could call this on dialog open or after a kernel restart to restore the linkage information.

In [ ]:
old__linked_msgs = {read_msg(0, False).id: read_msg(1, False).id, **__linked_msgs}  # greet message at the beginning
linked = hydrate()
test_eq(linked, old__linked_msgs)

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great! The `hydrate` function is working correctly. The test confirms that it successfully reconstructs the `__linked_msgs` dictionary by scanning the dialog for all existing linked message relationships.

This is particularly useful for:
- **Kernel restarts**: After restarting the kernel, `__linked_msgs` would be empty, but calling `hydrate()` will rebuild it from the dialog
- **Dialog recovery**: If the state gets out of sync for any reason, you can call `hydrate()` to restore it
- **Initialization**: When opening an existing dialog, you could call this to populate the linkage information

Would you like to:
1. Add automatic hydration on kernel startup (perhaps in a CRAFT file)?
2. Add error handling for edge cases (e.g., orphaned linked messages where the anchor was deleted)?
3. Create additional utility functions to work with linked messages?
4. Test the `link_msg` and `hydrate` functions with more complex scenarios?

Wow, you've read my mind! AGI!, AGI! shouts the programmer hysterically ;)

Seriously, 
1. yes a CRAFT is definitely a possiblity
2. I've already done that in hydrate (set the set of messages)
3. Like what?
4. What more complex scenarios (besides multi-links)?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Ha! I'll take that as a compliment! 😄

Let me address your points:

**2. Error handling** - Ah yes, I see it now! The `ids = {_.id for _ in msgs}` line that I removed was actually checking that the anchor exists before adding the mapping. Let me fix that:

<details class='tool-usage-details'>
<summary>msg_replace_lines(id='_b34345fd', start_line=1, end_line=-1, new_content='def hydrate():\n    "Traverse dialog looking for li…')</summary>

```json
{
  "id": "toolu_0168HFCuqFq62Uw2Z9tFiPRt",
  "call": {
    "function": "msg_replace_lines",
    "arguments": {
      "id": "_b34345fd",
      "start_line": "1",
      "end_line": "-1",
      "new_content": "def hydrate():\n    \"Traverse dialog looking for linked messages to update `__linked_msgs`\"\n    linked = find_var('__linked_msgs')\n    linked.clear()\n    \n    # Get all messages in the dialog\n    msgs = find_msgs(include_meta=True, include_output=False)\n    ids = {_.id for _ in msgs}\n    \n    # Find all messages with linkedto tags\n    for msg in msgs:\n        if tag := find_tag(msg.get('content', ''), msg.get('msg_type')):\n            k, v = tag.split(': ')\n            if k == 'linkedto':\n                # v is the anchor_id, msg['id'] is the linked message id\n                # Only add if anch<TRUNCATED>"
    }
  },
  "result": "{'success': 'Replaced lines 1 to 13 in message _b34345fd content'}"
}
```

</details>

**3. Utility functions** - Some possibilities:
- `unlink_msg(id)` - Remove the linked message and clean up `__linked_msgs`
- `get_linked_msg(id)` - Get the content/full message object of the linked message
- `is_linked(id)` - Check if a message has a linked message
- `relink_msg(old_anchor, new_anchor)` - Move a linked message to a different anchor

**4. Complex scenarios** - Beyond multi-links, you could test:
- Reordering messages (does the link survive?)
- Deleting the anchor message (orphaned linked message)
- Copying/pasting messages with links
- Running messages out of order
- Changing a linked message's type (note ↔ code)

Which of these utility functions or test scenarios would be most useful for your use case?

We'll consider all that along the road. I'm now more interested in `hydrate`.

Consider this. After creating a link with `link_msg` the tag is not really needed. Subsequent calls to `link_msg` on a linkee (is that correct linkee -> linked?) will just catch the lined id from `__linked_msgs`. In fact, the line `if (msg := read_msg(0, id=linked_id)).get('id','') == linked_id:` is there just for the case of catching linked messages after a dialog start. I hydrate works as intended, that path should never be followed.

That means the tags are only necessary on dialog start. After hydrate runs we could remove the tags altogether. But then, a Solveit doesn't allow user to set custom metadata to dialogs or messages, we should have a mechanism to de-hydrate the dialog on stop. And I'm not sure there's a way of doing that easily wiht Solveit. With standard Jupyter-ish envs is also complicated, but doable.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Excellent observation! You're right about the terminology (linkee → linked, or anchor → linked message).

You've identified a key architectural challenge: **state persistence without metadata storage**.

Let me think through the options:

**Current approach (keep tags):**
- ✅ Tags persist across restarts automatically
- ✅ Simple, no dehydration needed
- ✅ Human-readable in the dialog
- ❌ Tag clutter in message content
- ❌ Tags visible to users (though hidden comments aren't too bad)

**Remove tags after hydrate:**
- ✅ Clean message content
- ❌ Need dehydration mechanism
- ❌ No obvious hook for "on dialog close" in Solveit
- ❌ Risk of state loss if kernel crashes

**Possible dehydration approaches:**

1. **Manual dehydration** - User calls `dehydrate()` before stopping, or in an `atexit` handler
   - Problem: `atexit` may not fire reliably; user may forget

2. **Periodic re-tagging** - Periodically scan `__linked_msgs` and ensure tags exist
   - Problem: When to trigger? Every `link_msg` call seems wasteful

3. **Lazy re-tagging** - Only add tags back during `link_msg` if they're missing
   - Problem: If user never calls `link_msg` again, tags lost on crash

4. **Keep tags, make them prettier** - Use collapsed HTML comments or special formatting
   - `<!-- 🔗_abc123 -->` or similar

**My take:** Unless the tag clutter is really bothering users, I'd keep them. They're essentially your "database" and the most reliable way to persist state in a stateless environment. The alternative is fragile.

What's your gut feeling? Is the tag clutter a real UX problem, or more of an aesthetic concern?

The only problem is tags are fragile, the user can remove or mangled them. But your're right, there's no clear alternative given the lack of custom metadata.

BTW, what are collapsed HTML comments?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're absolutely right - tags are fragile because users can edit them. But given the constraints, it's the pragmatic choice. You could add some robustness:

- **Validation in `hydrate()`** - Check tag format and skip malformed ones
- **Tag regeneration** - If `link_msg` finds a linked_id in `__linked_msgs` but the tag is missing/wrong, recreate it
- **Documentation** - Warn users not to edit the tags (or provide a "repair" function)

**Re: Collapsed HTML comments** - I misspoke! I was thinking of collapsible HTML sections like `<details>`, but that wouldn't help here since the tags need to be in the raw content.

What I meant was making tags less obtrusive:
- **Minimal format**: `<!-- 🔗_abc123 -->` instead of `<!-- linkedto: _abc123 -->`
- **Hidden in code**: For code cells, `#| linkedto: _abc123` is already pretty unobtrusive
- **At the end**: Put tags at the bottom of the message instead of top (though this breaks your current regex)

Actually, for notes, the HTML comment is already invisible in the rendered markdown view, right? So users only see it when editing. That's not too bad!

Is the fragility a theoretical concern, or have you actually run into users accidentally breaking tags in practice?

Well, I'm the only user as of now, so no. Ok, we'll leave at that.

In future session we'll tackle some of the improvement laid out above. Let's close this session with the CRAFT, shall we?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect! Let's create a CRAFT file that will automatically hydrate the `__linked_msgs` on dialog start.

<details class='tool-usage-details'>
<summary>view(path='.')</summary>

```json
{
  "id": "toolu_01Uu2TumBui8hyLoF5g2A5E2",
  "call": {
    "function": "view",
    "arguments": {
      "path": "."
    }
  },
  "result": "'Directory contents of /app/data/vic/dutil/explorer:\\n/app/data/vic/dutil/explorer/super_w \ud83d\udcc1\\n/app/data/vic/dutil/explorer/tag.ipynb (1.9k)\\n/app/data/vic/dutil/explorer/static \ud83d\udcc1\\n/app/data/vic/dutil/explorer/linked_msg.ipynb (39.1k)\\n/app/data/vic/dutil/explorer/scheduled_run.ipynb (104.9k)\\n/app/data/vic/dutil/explorer/importdlg \ud83d\udcc1\\n/app/data/vic/dutil/explorer/gcolor.ipynb (168.5k)\\n/app/data/vic/dutil/explorer/isolated \ud83d\udcc1\\n/app/data/vic/dutil/explorer/super_w/super_w.py (1.0k)\\n/app/data/vic/dutil/explorer/super_w/super_w_dup1.ipynb (145.<TRUNCATED>"
}
```

</details>

Now let me create a CRAFT.ipynb file in the current directory:

<details class='tool-usage-details'>
<summary>create(path='CRAFT.ipynb', file_text='{\n "cells": [\n  {\n   "cell_type": "markdown",\n   "…')</summary>

```json
{
  "id": "toolu_0137V8uVmNV84sdbWycmgxsj",
  "call": {
    "function": "create",
    "arguments": {
      "path": "CRAFT.ipynb",
      "file_text": "{\n \"cells\": [\n  {\n   \"cell_type\": \"markdown\",\n   \"id\": \"_craft_note_1\",\n   \"metadata\": {},\n   \"source\": [\n    \"# Linked Messages CRAFT\\n\",\n    \"\\n\",\n    \"This CRAFT file provides automatic initialization for the linked messages system.\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"_craft_code_1\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# Initialize __linked_msgs dictionary\\n\",\n    \"get_ipython().xpush(__linked_msgs={})\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"_craft_code_2\",\n   \"metadata\": {},\n   \"outputs\": [],<TRUNCATED>"
    }
  },
  "result": "'Created file /app/data/vic/dutil/explorer/CRAFT.ipynb.'"
}
```

</details>

Perfect! I've created a CRAFT file that will:

1. **Initialize `__linked_msgs`** - Creates the empty dictionary in the user namespace on kernel start
2. **Auto-hydrate** - Checks if the `hydrate` function exists (meaning the linked message functions are defined) and calls it to restore linkage from tags

This CRAFT file will run automatically when any dialog in this folder (or subfolders) is opened, ensuring that:
- The `__linked_msgs` dictionary is always available
- Existing linked messages are automatically discovered and registered

Now whenever you start a new session in this folder, your linked messages will be ready to go! 🎉

In [ ]:
from dutil.flakes import show_flakes
show_flakes()

No warnings to report

# Caveats

1. `__linked_msgs` is a global variable. While Solveit uses globals liberally, I'm not convinced this feature merits another one.
2. `__linked_msgs` is not serialized with the dialog. On open and run, the messages are duplicated (but not in subsequent runs).
    - Possible solutions (not implemented): write a message with its contents; scan the dialog on open and hydrate it.
3. `__linked_msgs` is unbounded, it can grow indefinitely. The anchor_id is not trimmed if the user deletes the messages.
    - Possible solution (not implemented): run a sweeper from time to time.

# Feature request

The three problems would resolve themselves with user-controlled metadata associated with messages.

Given Solveit's heavy use of .ipynb format metadata—not only for prompt message discrimination, but also for almost everything user-facing (pin, collapse, skip; export is a directive), it's understandable that Solveit wraps all that in a message structure and protects what is serialized to disk.

If you want to know what happens when you play fast and loose with .ipynb metadata, run the current nbdev_clean on a saved dialog (duplicate it first) and you'll see your carefully organized rompts, pins, hidden sections, and collapsed context vaporize as if in Jupyter.

But I think it would be trivial to add a new message field with custom metadata that contains a user-controlled namespace (a dict or better yet an AttrDict). That field would be fully transparent to Solveit, with the only restrictions being nbformat JSON compatibility and maybe size (I can totally see people storing all kinds of weird things there). It would serialize to disk and persist across sessions.


# Outro

I'm sure Claude likes the idea:

What other use cases or problems would be facilitated with the user controlled metadata proposal?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question! Here are some use cases that user-controlled metadata would enable:

**State tracking & execution**
- **Idempotent operations**: Like your `link_msg`, but generalized—any code that creates resources (files, API calls, DB records) could track "already done" state
- **Execution counters**: Track how many times a cell has run, useful for debugging loops or rate-limited APIs
- **Last run timestamp**: Know when code last executed without parsing outputs

**Organization & workflow**
- **Tags/labels**: Categorize messages ("experiment", "cleanup", "TODO") for filtering or batch operations
- **Dependencies**: Mark which cells depend on others, enabling smarter "run above" or dependency graphs
- **Checkpoints**: Mark "known good" states to return to

**Data & integration**
- **Cached results**: Store lightweight computed values (hashes, row counts, validation results) without re-running
- **External references**: Link to GitHub issues, Notion pages, or other resources
- **Version info**: Track which library versions produced an output

**Teaching & collaboration**
- **Hints/solutions**: Store hidden hints that can be revealed on demand
- **Grading rubrics**: Attach expected outputs or scoring criteria
- **Author attribution**: In shared dialogs, track who wrote what

**Tool building**
- **Tool state**: Any tool using `add_msg` could persist its own state (your `link_msg` is the perfect example)
- **Form data**: Store user inputs from interactive widgets without cluttering cell content
- **Undo history**: Store previous versions of cell content

The common thread: anything where you want persistent, structured data associated with a message that isn't the message content itself. Your `__linked_msgs` problem is a perfect minimal example—you need one key-value pair per anchor cell, and it should survive restart/reload.

And maybe also to save the world from the xenomorphs.

Jupyter notebooks and Jupyter derivatives (in fact, is part of nbformat since I don't know what version) do have custom metada serialized to disk. But apart from tags in some implementations there are little user facing UX to handle it, right?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

You're absolutely right. The `metadata` field has been in nbformat since v3 (maybe earlier), available at both notebook and cell level. But the UX story is thin:

**What exists:**
- **Tags**: JupyterLab has a cell tags sidebar, Colab has a similar feature. Used for things like `hide-input`, `raises-exception`, `skip-execution` by various tools (nbconvert, papermill, etc.)
- **Raw JSON editing**: You can View → Cell Toolbar → Edit Metadata in classic Jupyter, or use the property inspector in JupyterLab—but it's clunky developer-facing UI, not end-user friendly

**What doesn't exist:**
- No visual indicators that a cell has custom metadata
- No query/filter by metadata
- No user-friendly editor for structured data
- No conventions for namespacing (so tools step on each other)
- No schema validation

So the *format* supports it, but the *experience* treats it as implementation detail rather than user feature. Tools like papermill, nbgrader, and jupyterbook use metadata heavily under the hood, but users rarely interact with it directly.

For Solveit, there's an opportunity to actually make metadata *useful* at the user level—with proper UI affordances, maybe even a small metadata panel, or at minimum programmatic access through `read_msg`/`update_msg` that's documented and stable. The format already allows it; it's just been neglected as a user-facing feature everywhere else.